In [1]:
import joblib
import pandas as pd

log_model = joblib.load('churn_model_logistic_regression.pkl')
scaler = joblib.load('churn_scaler.pkl')

df = pd.read_csv('telecom_ibm_feature_engineered.csv')
drop_cols = ['customerID', 'Name', 'Churn', 'Churn_Binary', 'TenureGroup', 
             'CustomerSegment', 'gender', 'Partner', 'Dependents', 
             'PhoneService', 'PaperlessBilling']
X = df.drop(columns=[c for c in drop_cols if c in df.columns])

print(X.columns.tolist())

['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Age', 'SupportTicketCount', 'Partner_Binary', 'Dependents_Binary', 'PhoneService_Binary', 'PaperlessBilling_Binary', 'Gender_Binary', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


In [2]:
# Var olan bir müşteriyi şablon olarak al, sonra istediğin değerleri değiştir
yeni_musteri = X.iloc[[0]].copy()

# Kendi senaryonu kur - örneğin:
yeni_musteri['tenure'] = 3            # yeni müşteri, 3 aydır abone
yeni_musteri['MonthlyCharges'] = 95.0  # yüksek aylık ücret
yeni_musteri['Age'] = 25
yeni_musteri['SupportTicketCount'] = 6  # çok fazla destek talebi açmış

# Tahmin yap
yeni_musteri_scaled = scaler.transform(yeni_musteri)
tahmin = log_model.predict(yeni_musteri_scaled)
olasilik = log_model.predict_proba(yeni_musteri_scaled)[:, 1]

print("Tahmin:", "Churn edecek" if tahmin[0] == 1 else "Kalacak")
print("Churn olasılığı: %{:.1f}".format(olasilik[0] * 100))

Tahmin: Kalacak
Churn olasılığı: %29.9


In [3]:
import numpy as np

yeni_musteri = pd.DataFrame([{
    'SeniorCitizen': 0,                    # 0=Hayır, 1=Evet
    'tenure': 2,                            # yeni müşteri, 2 ay
    'MonthlyCharges': 95.0,
    'TotalCharges': 190.0,
    'Age': 24,
    'SupportTicketCount': 7,                # çok şikayetçi
    'Partner_Binary': 0,
    'Dependents_Binary': 0,
    'PhoneService_Binary': 1,
    'PaperlessBilling_Binary': 1,
    'Gender_Binary': 1,                     # 1=Male, 0=Female
    'MultipleLines_No phone service': 0,
    'MultipleLines_Yes': 1,
    'InternetService_Fiber optic': 1,       # riskli grup (EDA'da görmüştük)
    'InternetService_No': 0,
    'OnlineSecurity_No internet service': 0,
    'OnlineSecurity_Yes': 0,                # güvenlik hizmeti YOK
    'OnlineBackup_No internet service': 0,
    'OnlineBackup_Yes': 0,
    'DeviceProtection_No internet service': 0,
    'DeviceProtection_Yes': 0,
    'TechSupport_No internet service': 0,
    'TechSupport_Yes': 0,                   # tech support YOK -> riskli
    'StreamingTV_No internet service': 0,
    'StreamingTV_Yes': 1,
    'StreamingMovies_No internet service': 0,
    'StreamingMovies_Yes': 1,
    'Contract_One year': 0,
    'Contract_Two year': 0,                 # ikisi de 0 -> Month-to-month (en riskli sözleşme)
    'PaymentMethod_Credit card (automatic)': 0,
    'PaymentMethod_Electronic check': 1,    # EDA'da yüksek churn ile ilişkiliydi
    'PaymentMethod_Mailed check': 0
}])

# Kolon sırasının X ile birebir aynı olduğundan emin ol
yeni_musteri = yeni_musteri[X.columns]

yeni_musteri_scaled = scaler.transform(yeni_musteri)
tahmin = log_model.predict(yeni_musteri_scaled)
olasilik = log_model.predict_proba(yeni_musteri_scaled)[:, 1]

print("Tahmin:", "Churn edecek" if tahmin[0] == 1 else "Kalacak")
print("Churn olasılığı: %{:.1f}".format(olasilik[0] * 100))

Tahmin: Churn edecek
Churn olasılığı: %81.1


In [5]:
def musteri_tahmin_et(tenure=24, MonthlyCharges=65, Age=40, SupportTicketCount=1, **kwargs):
    # Varsayılan "ortalama" bir müşteri profili
    default_musteri = {
        'SeniorCitizen': 0, 'tenure': tenure, 'MonthlyCharges': MonthlyCharges,
        'TotalCharges': tenure * MonthlyCharges, 'Age': Age, 'SupportTicketCount': SupportTicketCount,
        'Partner_Binary': 0, 'Dependents_Binary': 0, 'PhoneService_Binary': 1,
        'PaperlessBilling_Binary': 1, 'Gender_Binary': 1,
        'MultipleLines_No phone service': 0, 'MultipleLines_Yes': 0,
        'InternetService_Fiber optic': 0, 'InternetService_No': 0,
        'OnlineSecurity_No internet service': 0, 'OnlineSecurity_Yes': 1,
        'OnlineBackup_No internet service': 0, 'OnlineBackup_Yes': 1,
        'DeviceProtection_No internet service': 0, 'DeviceProtection_Yes': 1,
        'TechSupport_No internet service': 0, 'TechSupport_Yes': 1,
        'StreamingTV_No internet service': 0, 'StreamingTV_Yes': 0,
        'StreamingMovies_No internet service': 0, 'StreamingMovies_Yes': 0,
        'Contract_One year': 1, 'Contract_Two year': 0,
        'PaymentMethod_Credit card (automatic)': 1, 'PaymentMethod_Electronic check': 0,
        'PaymentMethod_Mailed check': 0
    }
    default_musteri.update(kwargs)  # senin verdiğin ekstra değerlerle üzerine yaz
    
    df_musteri = pd.DataFrame([default_musteri])[X.columns]
    scaled = scaler.transform(df_musteri)
    tahmin = log_model.predict(scaled)[0]
    olasilik = log_model.predict_proba(scaled)[:, 1][0]
    
    print(f"Tahmin: {'Churn edecek' if tahmin == 1 else 'Kalacak'} | Olasılık: %{olasilik*100:.1f}")
    return tahmin, olasilik

In [6]:
# Sadece ilgilendiğin değerleri gir, gerisi otomatik
musteri_tahmin_et(tenure=2, MonthlyCharges=95, SupportTicketCount=7, Contract_One_year=0)

musteri_tahmin_et(tenure=60, MonthlyCharges=40, SupportTicketCount=0)

Tahmin: Kalacak | Olasılık: %6.6
Tahmin: Kalacak | Olasılık: %1.6


(np.int64(0), np.float64(0.01552287955798177))